# Figure 1 — one window reduced to point sources

**In the paper:** Fig. 1; Sect. 2.2 (Eqs. 7 and 8); Appendix A.

One window, 18 samples along the scan, is reduced on its own to a list of
point sources with positions and error bars: its transit detection set
$D_j$ (Eq. 8).

| step | task | calls | paper |
|---|---|---|---|
| 1 | the instrument and the per-window thresholds $\Lambda_{\rm w}$ | `ins.Instrument.gaussian`, `ins.TOY_MARGINS` | Sect. 2.2, App. A |
| 2 | the design matrix $\mathbf A$ on the grid of trial positions | `inst.atoms`, `inst.lsf` (the instrument object) | Sect. 2.1, Eq. 4; Sect. 2.2 |
| 3 | the two windows of the figure: a resolved pair and a close blend | `sim.simulate_window`, `red.ladder_window`, `red.active_clusters`, `red.sigma_eta` | — |
| 4 | the non-negative solve: one noise-free source, then window A | `scipy.optimize.nnls` | Sect. 2.2, Eq. 7; App. A.1 |
| 5 | the reduction, stage by stage: (a) the non-negative solve on a grid of trial positions, no threshold (Eq. 7), its active nodes grouped into clusters (App. A.1); (b) one source, fitted continuously (App. A.2); (c) two sources, then three, each kept only if $\Delta\chi^2 > \Lambda_{\rm w}$ (App. A.2); (d) the formal error $\sigma_\eta$ of each position (Eq. 8) | `red.active_clusters`, `red.fit_k`, `red.sigma_eta`; in one call `red.ladder_window` | Sect. 2.2, App. A |
| 6 | the figure | `ins.render_curve`, `red.fit_positions` | Fig. 1 |

Every scanlib call carries the name of its module (`dyn.wls`, `joint.fit_joint`); the first code cell lists the
modules.

**Paper and code.** Grid node = atom; $\Lambda_{\rm w}$ = `margins`
($\Delta_1 = 11.1$ for a second source, $\Delta_2 = 10.8$ for a third;
derived in `auxil/sim_margins_lambda_w.ipynb`); $\Delta\chi^2$ = `D`.

In [ ]:
# the imports: numpy, matplotlib, scipy's nnls, and the scanlib modules; every scanlib call below carries its
# module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import simulation as sim      # synthetic scenes and their windows
from scanlib import reduction as red       # one window, or a campaign, reduced to detections

In [ ]:
# the instrument, defined once (Sect. 2.1): 18 samples of 59 mas along scan, a Gaussian LSF of FWHM 100 mas,
# and its per-window margins Delta_1, Delta_2 (App. A.2; derived in auxil/sim_margins_lambda_w)
FWHM_MAS = 100.0                           # the full width at half maximum of the LSF [mas]
PITCH_MAS = 59.0                           # the width of one sample along the scan [mas]
N_SAMPLES = 18                             # the number of samples of one window
# ins.Instrument.gaussian: the toy instrument -- its sample grid, its trial positions (atoms) and the matrix A of Eq. 4
inst = ins.Instrument.gaussian(fwhm_mas=FWHM_MAS, pitch_mas=PITCH_MAS, n_samples=N_SAMPLES, margins=ins.TOY_MARGINS)
# np.diff gives the step between neighbouring trial positions; [0] keeps the first (they are all equal)
NODE_STEP_MAS = float(np.diff(inst.atom_eta_mas)[0])   # the spacing of the trial positions, pitch / 8 [mas]
SNR_TOY = 100.0                            # the toy regime: S/N 100 on the total flux, sigma = sim.PRIMARY_FLUX / S/N

SAVE_PDF = False                           # True: write pdf/fig1_per_window_reduction.pdf

## 1. The instrument and the thresholds (Sect. 2.2, App. A)

The toy instrument: 18 samples of 59 mas; a Gaussian LSF of 100 mas FWHM;
white noise, S/N 100 for an isolated primary. The trial positions: a grid of
one eighth of a pixel, extending one FWHM beyond the samples (App. A.1).

In [ ]:
# the per-sample noise sigma_n the toy uses at S/N 100, and the instrument's numbers printed
sigma_n = sim.PRIMARY_FLUX / SNR_TOY
print(f"samples          {inst.sample_grid_mas.size}, pitch {PITCH_MAS:.1f} mas, "
      f"from {inst.sample_grid_mas[0]:.0f} to {inst.sample_grid_mas[-1]:.0f} mas")
print(f"LSF FWHM         {FWHM_MAS:.0f} mas")
print(f"atoms            {inst.atom_eta_mas.size}, step {NODE_STEP_MAS:.2f} mas, "
      f"from {inst.atom_eta_mas[0]:.0f} to {inst.atom_eta_mas[-1]:.0f} mas")
print(f"noise sigma      {sigma_n:.2f} per sample (S/N {SNR_TOY:.0f}, flux {sim.PRIMARY_FLUX:.0f})")

In [ ]:
# the per-window thresholds Lambda_w, and the paper's values asserted
margins = inst.margins                     # Lambda_w: [Delta_1, Delta_2], carried by the instrument
assert np.array_equal(np.round(margins, 1), [11.1, 10.8])          # App. A.2: 11.1 and 10.8
print(f"margins          Delta_1 = {margins[0]:.2f}, Delta_2 = {margins[1]:.2f}")

## 2. The design matrix (Sect. 2.1, Eq. 4; Sect. 2.2)

A window is $\vec s = \mathbf A(\boldsymbol\eta)\,\vec f + \vec\varepsilon$ (Eq. 4). On the grid of trial
positions $\eta_k$ (the nodes), column $k$ of $\mathbf A$ is the sampled profile of a unit-flux source at
$\eta_k$. For the toy, a Gaussian LSF integrated over a sample of width $p$ (midpoint rule):

$$A_{nk} = \mathcal P(x_n - \eta_k) = \frac{p}{\sqrt{2\pi}\,\sigma}\exp\!\left[-\frac{(x_n-\eta_k)^2}{2\sigma^2}\right],
\qquad \sigma = \frac{\rm FWHM}{2\sqrt{2\ln 2}},$$

with $x_n$ the centre of sample $n$ ($N = 18$) and $p = 59$ mas. Each column sums to one for a source
well inside the window: $A_{nk}$ is the fraction of its flux that sample $n$ collects.

**The block printed below** holds three columns, at nodes one sample pitch (59 mas) apart, and the five
samples near the centre.
- Down a column: the profile of a unit-flux source at that node. The two samples beside it collect 0.43
  and 0.44 of its flux.
- Across the columns: a node 59 mas further on gives the same column, moved down one row. $\mathbf A$ is
  one profile, shifted to each node.
- Each column sums to 0.99993: a source well inside the window puts all but 0.007 % of its flux into the
  18 samples.

In [ ]:
# Eq. 4 written out: the toy's design matrix A from the formula above, checked against the package's inst.atoms
sigma_lsf = FWHM_MAS / (2.0 * np.sqrt(2.0 * np.log(2.0)))           # the Gaussian's sigma [mas]
# x_n - eta_k for every sample n and node k, shape (N samples, M nodes): [:, None] makes the sample centres a
# column and [None, :] makes the nodes a row, so the difference is a table of every pair
offset = inst.sample_grid_mas[:, None] - inst.atom_eta_mas[None, :]
A = PITCH_MAS * np.exp(-0.5 * (offset / sigma_lsf) ** 2) / (np.sqrt(2.0 * np.pi) * sigma_lsf)
assert np.allclose(A, inst.atoms, rtol=1e-14, atol=0.0)                         # the formula is the package's matrix
assert np.array_equal(inst.lsf(offset), inst.atoms)                                 # inst.atoms is inst.lsf at x_n - eta_k
print(f"A: {inst.atoms.shape[0]} samples x {inst.atoms.shape[1]} nodes")

In [ ]:
# three of A's columns printed, and the three properties listed above asserted
# three columns of A, at nodes one sample pitch (59 mas = 8 node steps) apart: -58, 1 and 60 mas
# shown_nodes: the index k of the node nearest each of -59, 0 and 59 mas
shown_nodes = []
for eta_wanted in (-59.0, 0.0, 59.0):
    nearest_node = int(np.argmin(np.abs(inst.atom_eta_mas - eta_wanted)))
    shown_nodes.append(nearest_node)
shown_samples = np.arange(6, 11)                                           # the five samples nearest the centre
# the head of the table: one title per column (ljust and rjust pad a text with spaces to a fixed width)
head = "\nA_nk (x_n, eta_k in mas)".ljust(27)
for k in shown_nodes:                                                      # k counts the nodes (the columns of A)
    head = head + f"eta_k = {inst.atom_eta_mas[k]:.1f}".rjust(15)
print(head)
# one line per sample n (a row of A): its entry in each of the three columns
for n in shown_samples:
    line = f"x_n = {inst.sample_grid_mas[n]:.1f}".ljust(26)
    for k in shown_nodes:
        line = line + f"{inst.atoms[n, k]:15.4f}"
    print(line)
# the last line: each column summed over all the samples ([:, k] keeps every row of column k)
line = "column sum, all 18 samples".ljust(26)
for k in shown_nodes:
    line = line + f"{inst.atoms[:, k].sum():15.5f}"
print(line)

# 1. a column is one source's profile: its two largest entries are the samples beside the node (at 1.0 mas)
# shown_nodes[1] is the middle column; np.argsort orders the samples by their entry, and [-2:] keeps the last
# two, the largest
middle_column = inst.atoms[:, shown_nodes[1]]
assert set(np.argsort(middle_column)[-2:]) == {8, 9}                            # samples at -29.5 and +29.5 mas
# 2. one profile, shifted: 59 mas further on, the same column moved down one row, value for value
# shown_nodes[:-1] is every column but the last and shown_nodes[1:] every column but the first, so the loop
# takes each column with its neighbour to the right
for k_left, k_right in zip(shown_nodes[:-1], shown_nodes[1:]):
    # [1:, k_right] keeps the right column without its first row; [:-1, k_left] keeps the left column without
    # its last row
    assert np.array_equal(inst.atoms[1:, k_right], inst.atoms[:-1, k_left])
# 3. the flux is kept: a column sums to one for a source well inside the window (all(...) is True when the test
# holds for each of the three columns)
assert all(abs(inst.atoms[:, k].sum() - 1.0) < 1e-3 for k in shown_nodes)

## 3. The two windows of the figure

Row A: a resolved pair, 150 mas, flux ratio 0.2. Row B: a close blend, 45 mas,
equal fluxes. Each window is drawn from one seeded generator until the
reduction selects two sources, its typical outcome. This uses the truth and is
for presentation only.

In [ ]:
# the two scenes of the figure: a resolved pair (row A) and a close blend (row B)
# SCENES: a table with one row per scene. Each row holds four things: the label drawn and printed, the true
# positions [mas], the true fluxes [flux per sample], and the number of sources the ladder should select
SCENES = [
    ("A  resolved\n150 mas,  q = 0.2", [0.0, 150.0], [sim.PRIMARY_FLUX, 200.0], 2),
    ("B  deformed, recovered\n45 mas,  q = 1", [0.0, 45.0], [sim.PRIMARY_FLUX, sim.PRIMARY_FLUX], 2),
]

In [ ]:
# truth: each scene is drawn from one generator until the one-call reduction gives its typical outcome (two
# sources); what sections 4 and 5 need is kept in `rows`
# the generator of the noise. Its seed is one number, 7. The two scenes share the generator: the draws of scene
# B follow those of scene A, so scene B's window depends on how many draws scene A took
rng_scenes = np.random.default_rng(7)
MAX_DRAWS = 400                            # the cap on the draws of one scene
# rows: one entry per scene, (label, true positions [mas], true fluxes, reduced), where reduced is a dictionary
# of what the reduction gave for the kept window
rows = []
for label, true_eta, true_flux, expected_k in SCENES:
    # step 1, draw windows until the reduction selects expected_k sources. n_draw counts the draws; the windows
    # drawn before the kept one are dropped
    for n_draw in range(1, MAX_DRAWS + 1):
        sigma = sigma_n
        # sim.simulate_window: the scene through the LSF, plus noise
        samples = sim.simulate_window(true_eta, true_flux, inst, rng_scenes, sigma)
        # red.ladder_window: the whole reduction of section 5 in one call -- the selected positions and fluxes,
        # their number k, and diagnostics (the active nodes, the Delta chi^2 of each rung)
        eta, flux, k, diag = red.ladder_window(samples, inst, sigma)
        if k == expected_k:                                         # keep the first draw with the typical outcome
            break
    else:
        # the else of a for loop runs when the loop ends without a break: no draw gave the outcome
        raise RuntimeError("scene outcome not found")

    # step 2, the grid solution of the kept window. red.active_clusters: the non-negative grid solution (one
    # flux per node) and its clusters (stage (a) of section 5; the clusters are dropped here, the name _). The
    # grid solution is drawn in the left panels
    grid_flux, _ = red.active_clusters(samples, inst)

    # step 3, red.sigma_eta: the Fisher error [mas] of each selected position (Eq. 8)
    sig_eta = red.sigma_eta(eta, flux, inst, sigma)

    # step 4, the record of this window: the samples, the noise sigma, the grid solution, the selected positions
    # [mas] and fluxes, their number K, their errors [mas], the positions of the active nodes [mas], and the
    # Delta chi^2 of each rung
    reduced = dict(samples=samples, sigma=sigma, grid_flux=grid_flux, eta=eta, flux=flux, K=k,
                   sig_eta=sig_eta, active_eta=diag["active_eta"], delta_chi2=diag["delta_chi2"])
    rows.append((label, true_eta, true_flux, reduced))
    # label.splitlines()[0] keeps the first line of the label (the row's name) and omits the scene's numbers
    print(f"{label.splitlines()[0]:26s} K = {k} after {n_draw} draw(s)")

## 4. The non-negative solve (Sect. 2.2, Eq. 7; App. A.1)

$$\hat{\vec f} = \arg\min_{\vec f \ge 0}\ \lVert \vec s - \mathbf A\,\vec f\rVert^2,$$

one flux per node, with no threshold (with one $\sigma_n$ for every sample, the weights drop out).
**A simplified case:** one noise-free source between two nodes. Its flux goes to the nodes beside it,
not to its true position (Sect. 2.2): the grid solution only starts the continuous fits.
**Window A:** the package's solve is this one. Without the constraint, the least-squares solution
of the 18 × 144 system spreads about the same light over most nodes, with both signs, so that
neighbouring columns cancel; it fits the noise exactly (Sect. 2.2).

In [ ]:
# section 4: the non-negative solve (scipy's nnls, Eq. 7) on a simplified case
# the simplified case: a noise-free source at 4.7 mas, between the nodes at 1.0 and 8.4 mas
ETA_ONE_MAS = 4.7                          # the position of the one source [mas]
ACTIVE_FRACTION = 1e-9                     # a node is active above this fraction of the brightest node's flux (A.1)
samples_one = sim.PRIMARY_FLUX * inst.lsf(inst.sample_grid_mas - ETA_ONE_MAS)
# nnls returns (the solution, the norm of the residual); we keep the solution and omit the norm
grid_flux_one, _ = nnls(inst.atoms, samples_one)
# np.flatnonzero gives the indices of the nodes where the test holds: the active nodes
active_nodes = np.flatnonzero(grid_flux_one > ACTIVE_FRACTION * grid_flux_one.max())
# the printed line: each active node's position and flux, then the summed flux
node_texts = []
for k in active_nodes:                                                     # k counts the active nodes
    node_texts.append(f"{inst.atom_eta_mas[k]:.1f} mas: {grid_flux_one[k]:.1f}")
print("one source at 4.7 mas, flux 1000 -> nodes " + ", ".join(node_texts)
      + f"; summed flux {grid_flux_one.sum():.1f}")
# two active nodes, one on each side of the source
first_active = inst.atom_eta_mas[active_nodes[0]]
second_active = inst.atom_eta_mas[active_nodes[1]]
assert active_nodes.size == 2 and first_active < ETA_ONE_MAS < second_active

In [ ]:
# window A of the figure: the same solve; red.active_clusters (section 5) returns this solution
# rows[0] is window A; its [3] is the dictionary of what the reduction gave (the label and the truth are omitted)
reduced_A = rows[0][3]
samples_A = reduced_A["samples"]
grid_flux_A, _ = nnls(inst.atoms, samples_A)                              # the norm of the residual is dropped (the _)
assert np.array_equal(grid_flux_A, reduced_A["grid_flux"])
# the same A and window without f >= 0: of the many exact solutions, lstsq returns the one of least sum f^2,
# f = A^T c, a sum of the 18 rows of A, each a Gaussian in eta_k: smooth across the nodes and of both signs
# [0] keeps lstsq's solution and omits the residuals, the rank and the singular values
grid_flux_free = np.linalg.lstsq(inst.atoms, samples_A, rcond=None)[0]
n_active_A = np.sum(grid_flux_A > ACTIVE_FRACTION * grid_flux_A.max())    # the number of active nodes
n_negative_free = np.sum(grid_flux_free < 0)                              # the number of negative fluxes
print(f"window A: non-negative solve, {n_active_A} active nodes, fluxes 0 to {grid_flux_A.max():.0f}; "
      f"without the constraint {n_negative_free} of {grid_flux_free.size} fluxes negative, "
      f"from {grid_flux_free.min():.0f} to {grid_flux_free.max():.0f}")

In [ ]:
# the same light, distributed differently: the summed flux, its share near each true source, and the fit
sigma_A = reduced_A["sigma"]
print("                  summed flux | near 0 mas | near 150 mas | nodes |f|>1 | chi^2 (18 samples)    "
      "truth: 1000 at 0, 200 at 150")
# the two boolean masks keep the nodes within 60 mas of a true source (0 and 150 mas) and omit the others
near_0 = np.abs(inst.atom_eta_mas) <= 60
near_150 = np.abs(inst.atom_eta_mas - 150) <= 60
for name, solution in (("f >= 0", grid_flux_A), ("no constraint", grid_flux_free)):
    # inst.atoms @ solution is the model A f; chi^2 is the summed squared residual over sigma^2
    chi2 = np.sum((samples_A - inst.atoms @ solution) ** 2) / sigma_A ** 2
    # np.abs(solution) > 1 counts the nodes that hold more than one flux unit, of either sign
    n_lit = np.sum(np.abs(solution) > 1)
    print(f"  {name:14s} {solution.sum():12.0f} | {solution[near_0].sum():10.0f} | {solution[near_150].sum():12.0f} | "
          f"{n_lit:11d} | {chi2:6.2f}")
# 144 unknowns, 18 samples: the noise is fitted exactly
assert np.sum((samples_A - inst.atoms @ grid_flux_free) ** 2) < 1e-12 * np.sum(samples_A ** 2)

In [ ]:
# the figure of this section: (a) A as an image, (b) three of its columns, (c) window A solved with and
# without f >= 0
# the paper's style: grayscale, identity by line style and marker, ticks inward
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in"}
active_A = grid_flux_A > ACTIVE_FRACTION * grid_flux_A.max()             # a mask: the active nodes only
with plt.rc_context(PAPER_STYLE):
    fig_nnls, ax = plt.subplots(1, 3, figsize=(7.5, 2.4))                 # ax: the three panels, ax[0] to ax[2]
    # (a) A as an image: white 0, black the largest entry
    ax[0].imshow(inst.atoms, aspect="auto", cmap="Greys", interpolation="nearest",
                 extent=[inst.atom_eta_mas[0], inst.atom_eta_mas[-1], inst.sample_grid_mas[-1], inst.sample_grid_mas[0]])
    ax[0].set_xlabel(r"node $\eta_k$ [mas]")
    ax[0].set_ylabel(r"sample $x_n$ [mas]")
    # (b) three columns of A: the sampled profile of a unit-flux source at three nodes
    for k, line_style in zip(shown_nodes, ("--", "-", ":")):
        ax[1].step(inst.sample_grid_mas, inst.atoms[:, k], where="mid", color="k", lw=0.9, ls=line_style,
                   label=rf"$\eta_k$ = {inst.atom_eta_mas[k]:.0f} mas")
    ax[1].set_xlim(-300, 300)
    ax[1].set_ylim(0.0, 0.68)                                             # headroom: the legend sits above the curves
    ax[1].set_xlabel(r"sample $x_n$ [mas]")
    ax[1].set_ylabel(r"$A_{nk}$")
    ax[1].legend(fontsize=7, frameon=False, loc="upper right", handlelength=1.8)
    # (c) window A solved, one flux per node for both: without the constraint (gray stems, open markers, all
    #     144 nodes) and with f >= 0 (black stems, filled markers, the active nodes)
    # stem returns the markers, the stems and the baseline (dropped, the name _)
    markers, stems, _ = ax[2].stem(inst.atom_eta_mas, grid_flux_free, basefmt=" ", label="no constraint")
    plt.setp(stems, color="0.6", lw=0.6)
    plt.setp(markers, color="0.6", ms=2.5, mfc="white", mew=0.6)
    markers, stems, _ = ax[2].stem(inst.atom_eta_mas[active_A], grid_flux_A[active_A], basefmt=" ",
                                   label=r"$\vec f \geq 0$")
    plt.setp(stems, color="k", lw=0.9)
    plt.setp(markers, color="k", ms=3.0)
    ax[2].axhline(0, color="0.8", lw=0.5)
    ax[2].set_xlim(-300, 350)
    ax[2].set_xlabel(r"node $\eta_k$ [mas]")
    ax[2].set_ylabel("flux per node")
    ax[2].legend(fontsize=7, frameon=False, loc="upper right")
    for panel, letter in zip(ax, "abc"):
        panel.text(0.05, 0.97, f"({letter})", transform=panel.transAxes, va="top", ha="left", fontsize=9,
                   fontweight="bold", bbox=dict(fc="white", ec="none", pad=1.0))
    fig_nnls.tight_layout(pad=0.4, w_pad=0.8)
    plt.show()

## 5. The reduction, stage by stage (Sect. 2.2, App. A)

Stages (a)–(d) of the outline, with the package's functions. The last lines
check that the one-call `red.ladder_window` of section 3 made the same fits and
the same decisions.

In [ ]:
# section 5: the reduction of both windows written out, stages (a) to (d); it must repeat red.ladder_window's fits
# and decisions exactly
for label, true_eta, true_flux, reduced in rows:
    samples = reduced["samples"]
    sigma = reduced["sigma"]
    print(f"--- {label.splitlines()[0]}  (truth eta {true_eta}, fluxes {true_flux})")
    # (a) non-negative solve on the grid (Eq. 7); active atoms closer than one pixel form a cluster (App. A.1)
    grid_flux, clusters = red.active_clusters(samples, inst)
    # each cluster is the list of its nodes' indices; its edges are its lowest and highest node position, rounded
    # to 1 mas
    cluster_edges = []
    for nodes in clusters:
        lowest = round(float(inst.atom_eta_mas[nodes].min()))
        highest = round(float(inst.atom_eta_mas[nodes].max()))
        cluster_edges.append((lowest, highest))
    print("  clusters of active atoms [mas]  "
          f"{cluster_edges}")
    # (b) one source: red.fit_k, the continuous least-squares fit of k sources (started from the clusters, or from
    #     `previous`), returns (positions, fluxes, residual sum of squares)
    # fits: the fits made so far, in order: fits[0] one source, fits[1] two sources, fits[2] three
    fits = [red.fit_k(samples, 1, inst)]
    decided = 1                                                              # the number of sources accepted so far
    # (c) add a source while the drop in chi^2 exceeds the threshold; the K-source fit also starts from the
    #     (K-1)-source one, so the fits are nested and D is never negative (App. A.2)
    for k in (2, 3):                                                         # k: the number of sources tried
        # fits[-1] is the last fit; [:2] keeps its positions and fluxes and omits its residual sum of squares
        previous_fit = fits[-1][:2]
        fits.append(red.fit_k(samples, k, inst, previous=previous_fit))
        # fits[-2] is now the fit with k - 1 sources and fits[-1] the fit with k; [2] is the residual sum of squares
        D = (fits[-2][2] - fits[-1][2]) / sigma ** 2                         # Delta chi^2
        threshold = margins[k - 2]                                       # Delta_1 for the 2nd source, Delta_2 for the 3rd
        accepted = D > threshold
        if accepted:
            decision = "accept"
        else:
            decision = "stop"
        print(f"  {k - 1} -> {k} sources: D_{k - 1} = {D:7.2f} vs Delta_{k - 1} = "
              f"{threshold:.2f}  -> {decision}")
        if not accepted:                                                     # the ladder stops at its first failure
            break
        decided = k
    # (d) the selected fit and the Fisher error of each position: the detection set D_j (Eq. 8)
    # fits[decided - 1] is the fit with `decided` sources; its residual sum of squares is dropped (the name _)
    eta, flux, _ = fits[decided - 1]
    sig_eta = red.sigma_eta(eta, flux, inst, sigma)
    for eta_one, flux_one, sig_one in zip(eta, flux, sig_eta):
        print(f"  source at {eta_one:7.1f} +/- {sig_one:4.1f} mas, flux {flux_one:7.1f}")
    # the stage-by-stage result equals red.ladder_window's, fit for fit and decision for decision
    assert decided == reduced["K"] and np.array_equal(eta, reduced["eta"]) and np.array_equal(flux, reduced["flux"])
    # the Delta chi^2 of every rung made here (i counts the rungs: fit i against fit i + 1)
    delta_chi2_by_hand = []
    for i in range(len(fits) - 1):
        delta_chi2_by_hand.append((fits[i][2] - fits[i + 1][2]) / sigma ** 2)
    assert np.array_equal(np.array(delta_chi2_by_hand), reduced["delta_chi2"])
print("red.ladder_window made the same fits and the same decisions")

## 6. The figure (Fig. 1)

Left, (a) and (c): the samples (steps), the noise-free profile (gray), the true
positions (shaded bands), the grid fluxes (stems) and the clusters (numbered
bars, ranked by summed flux). Right, (b) and (d): the selected two-source fit
(solid) and the best single source (dashed). Plotting only.

In [ ]:
# Figure 1: one row per window, the samples and the grid solution on the left, the selected fit on the right.
# Plotting only
XLIM = (-280.0, 320.0)                     # the drawn range of eta [mas]
ETA_FINE = np.linspace(*XLIM, 400)         # the eta of the drawn curves [mas]
GRAY_TRUTH = "0.55"                        # the gray of the noise-free profile


def cluster_ranges(active_eta, link=PITCH_MAS, step=NODE_STEP_MAS):
    """[(lo, hi)] [mas] of the clusters of active atoms, linked as red.active_clusters
    links them (closer than one sample) and widened by half an atom step.

    Inputs: the positions of the active atoms [mas], in increasing order; the largest gap inside one cluster
    [mas]; the atom step [mas]."""
    eta = np.asarray(active_eta, dtype=float)
    if eta.size == 0:
        return []
    breaks = np.flatnonzero(np.diff(eta) > link * (1.0 + 1e-9))        # a gap wider than one pixel starts a new cluster
    # np.r_ joins its entries into one array: a cluster starts at node 0 or after a break, and stops at a break
    # or at the last node
    starts = np.r_[0, breaks + 1]
    stops = np.r_[breaks, eta.size - 1]
    ranges = []
    for i, j in zip(starts, stops):                                    # i, j: the first and last atom of a cluster
        ranges.append((eta[i] - step / 2, eta[j] + step / 2))
    return ranges


plt.rcParams.update({"font.size": 10.0, "axes.linewidth": 0.5,
                     "xtick.labelsize": 9.0, "ytick.labelsize": 9.0})
psf_peak = float(inst.lsf(np.array([0.0]))[0])          # atom amplitude -> peak flux per sample
letters = iter("abcdefgh")                              # the panels' letters, taken one at a time by next(letters)

fig, axes = plt.subplots(len(rows), 2, figsize=(7.5, 3 * len(rows)),
                         sharex=True, sharey="row", squeeze=False)
# one row of panels per window: ax_w (left) the window, ax_f (right) the fit
for (label, true_eta, true_flux, reduced), (ax_w, ax_f) in zip(rows, axes):
    # left: true positions as bands, the noise-free truth, the samples, the grid solution
    for true_position in true_eta:
        ax_w.axvline(true_position, ls="-", lw=7, color="tomato", zorder=1, alpha=0.2)
    # ins.render_curve: the noise-free window as a continuous curve of eta, the sources' LSFs summed
    ax_w.plot(ETA_FINE, ins.render_curve(ETA_FINE, true_eta, true_flux, inst), ls="-",
              lw=0.9, color=GRAY_TRUTH, zorder=2)
    ax_w.step(inst.sample_grid_mas, reduced["samples"], where="mid", color="k", linewidth=0.6, zorder=3)
    active = reduced["grid_flux"] > 1e-6                        # draw only the atoms the solve made active
    # stem returns the markers, the stems and the baseline (not used)
    markers, stems, baseline = ax_w.stem(inst.atom_eta_mas[active], reduced["grid_flux"][active] * psf_peak,
                                         basefmt=" ")
    plt.setp(stems, color="k", lw=0.9)
    plt.setp(markers, color="k", ms=3.0)
    # the clusters inside the frame, as bars ranked by summed grid amplitude
    # ranges: the (lo, hi) of each cluster that overlaps the drawn range
    ranges = []
    for one_range in cluster_ranges(reduced["active_eta"]):
        if one_range[1] >= XLIM[0] and one_range[0] <= XLIM[1]:
            ranges.append(one_range)
    # sums: the summed grid flux of the atoms inside each cluster
    sums = []
    for lo, hi in ranges:
        inside = (inst.atom_eta_mas >= lo) & (inst.atom_eta_mas <= hi)
        sums.append(reduced["grid_flux"][inside].sum())
    order = np.argsort(-np.asarray(sums), kind="stable")       # rank 1 = the cluster with the most flux
    bar_transform = ax_w.get_xaxis_transform()                  # x in mas, y in axes fraction: bars at the bottom
    for rank, j in enumerate(order, start=1):                   # j: the cluster's place in ranges
        lo, hi = ranges[j]
        ax_w.plot([lo - 4, hi + 4], [0.045, 0.045], transform=bar_transform, color="k", lw=2.2,
                  solid_capstyle="butt", zorder=5)
        ax_w.text(0.5 * (lo + hi), 0.035, str(rank), transform=bar_transform, ha="center",
                  va="top", fontsize=7.5, zorder=5, clip_on=True)
    ax_w.set_ylabel(r"Flux (arbitrary units)", fontsize=14, labelpad=4)

    # right: the samples, the selected fit and the best single source
    ax_f.step(inst.sample_grid_mas, reduced["samples"], where="mid", color="k", linewidth=0.6, zorder=3)
    ax_f.plot(ETA_FINE, ins.render_curve(ETA_FINE, reduced["eta"], reduced["flux"], inst), "-",
              lw=1.3, color="k", zorder=4)
    if reduced["eta"].size >= 2:
        # the best single source: red.fit_positions refines the positions from a start (here the brighter fitted
        # source) and returns the positions and fluxes
        brighter = np.argmax(reduced["flux"])                   # the index of the brighter fitted source
        eta_single, flux_single = red.fit_positions(reduced["samples"], [reduced["eta"][brighter]], inst)
        ax_f.plot(ETA_FINE, ins.render_curve(ETA_FINE, eta_single, flux_single, inst), ls=(0, (4, 2)),
                  lw=1.0, color="0.45", zorder=3)

    for ax in (ax_w, ax_f):
        ax.text(0.05, 0.97, f"({next(letters)})", transform=ax.transAxes,
                va="top", ha="left", fontsize=9, fontweight="bold")

for ax in axes.ravel():
    ax.set_xlim(*XLIM)
    ax.set_xticks([-200, 0, 200])
    ax.tick_params(direction="in")
# axes[:-1]: every row but the last; axes[:, 1]: the right column; axes[-1]: the last row
for ax in axes[:-1].ravel():
    ax.tick_params(labelbottom=False)
for ax in axes[:, 1]:
    ax.tick_params(labelleft=False)
for ax in axes[-1]:
    ax.set_xlabel(r"along-scan $\eta$  [mas]", fontsize=14)
fig.tight_layout(pad=0.4, h_pad=0.5, w_pad=0.5)
plt.show()

In [ ]:
# write the PDF, only when SAVE_PDF is True
if SAVE_PDF:
    Path("pdf").mkdir(exist_ok=True)
    fig.savefig(Path("pdf") / "fig1_per_window_reduction.pdf", bbox_inches="tight")
    print("wrote pdf/fig1_per_window_reduction.pdf")
else:
    print("SAVE_PDF is False: the PDF is not written")